# EP3: Buffon's needle on the GPU and Amdahl's law

EP1's Monte Carlo estimate of π, accelerated with CUDA, with Amdahl's law measured over the GPU's
streaming multiprocessors (SMs).

- Code: https://github.com/welt00n/aga511/tree/main/EP3
- `buffon.cu`: the engine (one experiment per call, on the GPU or, as in EP1, on one CPU core)
- `buffon.py`: the experiments, the fits and the graphs

**To run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

## The machine

In [ ]:
!nvidia-smi

## The code

In [ ]:
!git clone -q https://github.com/welt00n/aga511.git /content/aga511 2>/dev/null || git -C /content/aga511 pull -q
%cd /content/aga511/EP3

In [ ]:
!cat buffon.cu

In [ ]:
!cat buffon.py

## The engine: one experiment per call

`-n` needles, `-s` seed, `-p` precision (64, 32 or 16), `-m` number of SMs (0 = all), `-c` one CPU core.
The same seed gives the same π on any number of SMs.

In [ ]:
!nvcc buffon.cu -o buffon -O2 -arch=native
!./buffon -i
!./buffon -n 1e8 -s 42
!./buffon -n 1e8 -s 42 -m 1
!./buffon -n 1e7 -s 42 -c

## Results of the full experiment

Run on the author's laptop (NVIDIA GeForce RTX 5070 Laptop GPU, Intel Core Ultra 7 255HX):
GPU and CPU in FP64 for 30 minutes each, FP32 and FP16 for 10 minutes each.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

def show_run(folder):
    print((folder / "summary.txt").read_text())
    for name in ("error.png", "amdahl.png", "prediction.png"):
        if (folder / name).exists():
            display(Image(str(folder / name), width=1100))

def laptop_run(name):
    return sorted(Path("runs").glob(f"2026-11-08_*-{name}"))[0]

report = sorted(Path("runs").glob("2026-11-08_*-report"))[-1]
print((report / "report.txt").read_text())
display(Image(str(report / "report.png"), width=1100))

### GPU, double precision (FP64)

In [ ]:
show_run(laptop_run("gpu-fp64"))

### CPU, double precision (FP64): EP1

In [ ]:
show_run(laptop_run("cpu-fp64"))

### The other precisions

In [ ]:
for name in ("gpu-fp32", "gpu-fp16", "cpu-fp32", "cpu-fp16"):
    show_run(laptop_run(name))

## A run on Colab's GPU

A shorter run (5 minutes) on the GPU Colab provides, to show the code works here too.

In [ ]:
!python buffon.py --budget 5m

In [ ]:
colab_run = sorted(Path("runs").glob("*-gpu-fp64"))[-1]
show_run(colab_run)

## Discussion

- Does the real error behave as expected? (real error vs experimental error vs N)
- How many needles and how much time for 14 decimals, on the CPU and on the GPU?
- How does the speedup grow with the number of SMs? What is p, and what limits the speedup?
- Does p depend on the size of the problem?
- Does the run time grow as O(N)?
- How do FP64, FP32 and FP16 compare, in speed and in accuracy?
- What changed from EP1 (random number generator, π constant, fixed-slope extrapolation)?